# Unit 08 · Power, MDE and duration · Exercise

<a href="https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/notebooks/unit-08-power-mde-exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Finance agrees to ship a new checkout if it lifts conversion from 12% by at
least **half a percentage point** - below that, the build cost is not worth it.
The team opens an online sample-size calculator. It comes pre-filled with a
"minimum detectable effect" of 1 point, says 17,000 users per group, and
everyone is happy: the test fits in two weeks. What did they just give up?

**After this notebook you can:** size a test from a business threshold, show what the calculator default misses, turn users into weeks, and check power by simulation.

**Time:** about 15 seconds to run in Colab. No accounts, no API keys, no files to upload.

## Without code

You can do this exercise without writing Python. Open the
[worked demo](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/solutions/unit-08-power-mde-demo.ipynb) (its outputs are saved), then answer the
questions in the **Your answer** section at the end of this notebook in
plain words. What you should find:

1. The business MDE (0.5 points) needs about **67,500 users per arm**.
2. The default-sized test (1-point MDE, about 17,000 per arm) has only about **30%** power for a true 0.5-point lift.
3. At 20,000 users a week, the business-sized test takes about **7 weeks**.
4. Simulating the business-sized test, about **80%** of runs are significant.

## 1. The question

Finance ships above a 0.5-point lift on a 12% baseline. Size the test, show
what the 1-point default would miss, and check your answer by simulation.

## 2. Setup

The next cell loads the libraries and fixes the random seed, so everyone gets
the same numbers. Colab already has every library this notebook needs. If an
import ever fails, delete the `#` in front of `%pip` and run the cell again.

In [ ]:
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change it and re-run: the numbers move, the lesson should not
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The inputs

Given: the baseline, alpha, power target, traffic, and a `power()` helper. You
will write the sample-size formula yourself.

In [ ]:
p0 = 0.12                      # baseline conversion
alpha = 0.05                   # false-positive rate we accept
power_target = 0.80            # chance of catching a real effect of the MDE size
z_alpha = stats.norm.ppf(1 - alpha / 2)
z_power = stats.norm.ppf(power_target)

def power(true_lift, n, p0=p0):
    '''Chance a test with n users per arm comes out significant when the true lift is true_lift.'''
    p1 = p0 + true_lift
    se = np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)) / n)
    return stats.norm.cdf(true_lift / se - z_alpha)

USERS_PER_WEEK = 20_000        # users entering the test each week, both arms together

business_mde = 0.005   # the smallest lift finance would act on
default_mde = 0.010    # the calculator's pre-filled value

**How the exercise works.** Each `TODO` cell tells you what to compute. Replace
the `None` with your code and run the cell. The `assert` lines below it check
your answer: if one fails, its message tells you what to look at. The data
cells are given - you only write the parts that carry the idea.

## 4. TODO 1 - sample size from the business MDE

Compute `n_business`, users per arm, with

`n = (z_alpha + z_power)^2 * (p0(1-p0) + p1(1-p1)) / MDE^2`, where `p1 = p0 + MDE`.

Round **up** to a whole number of users. Do the same for `n_default`.

In [ ]:
# TODO: the formula above, for business_mde and for default_mde (round up)
n_business = None
n_default = None

assert n_business is not None and n_default is not None, 'Replace both Nones'
print(f'Business MDE: {n_business:,} per arm   Default MDE: {n_default:,} per arm')
assert 60_000 <= n_business <= 72_000, 'Check the formula - p1 is p0 + MDE, and MDE is squared'
assert 3.5 < n_business / n_default < 4.5, 'Halving the MDE should roughly quadruple n'

## 5. TODO 2 - what the default misses

Use the given `power(true_lift, n)` to compute `missed_power`: the power of the
**default-sized** test when the true lift is the **business** threshold, 0.5 points.

In [ ]:
# TODO: power of the default-sized test for a true lift of 0.005
missed_power = None

assert missed_power is not None, 'Replace None'
print(f'Power of the default-sized test at a true 0.5-point lift: {missed_power:.0%}')
assert missed_power < 0.4, 'Use n_default and a true lift of business_mde'

## 5b. TODO 3 - users into weeks

Both arms together need `2 * n_business` users; `USERS_PER_WEEK` arrive each
week. Compute `weeks` as a whole number of weeks, rounded **up**.

In [ ]:
# TODO: whole weeks needed for the business-sized test
weeks = None

assert weeks is not None, 'Replace None'
print(f'Weeks needed: {weeks}')
assert weeks == 7, 'Total users over users per week, rounded up'

## 5c. TODO 4 - check power by simulation

Simulate 1,000 experiments with `n_business` users per arm and a true lift of
exactly `business_mde`. In each, compute the difference in conversion rates and
its standard error, and call it significant if `|diff| / SE > z_alpha`.
`sim_power` is the share of significant runs.

Tip: `np.random.binomial(n_business, p0, 1000) / n_business` gives 1,000
simulated control conversion rates in one line.

In [ ]:
# TODO: 1,000 simulated experiments, share significant
sim_power = None

assert sim_power is not None, 'Replace None'
print(f'Simulated power: {sim_power:.0%}')
assert 0.72 < sim_power < 0.88, 'Should be close to the 80% you designed for'

## Your answer

Double-click this cell and write your answers under each question. One or two
sentences each is enough - we grade the reasoning, not the length.

**1.** In one sentence to a finance lead: why must the business threshold come before the calculator?

**2.** Seven weeks is too long for your team. Name two levers you could pull, and what each one costs.

**3.** What is the smallest effect worth acting on in *your* project, and why? (This feeds Task 08.)

## Hints

<details>
<summary>Open a hint only after you have tried</summary>

- `np.ceil(x)` rounds up; wrap it in `int(...)`.
- `missed_power = power(business_mde, n_default)`.
- Simulation: draw control and treatment rates for all 1,000 runs at once, then work with arrays - no loop needed.

</details>

## Spoiler: full solution

Looking before you have tried costs you the exercise - the struggle is where
the learning happens.

<details>
<summary>Show the solution</summary>

```python
def n_for(mde):
    p1 = p0 + mde
    return int(np.ceil((z_alpha + z_power) ** 2 * (p0 * (1 - p0) + p1 * (1 - p1)) / mde ** 2))

n_business = n_for(business_mde)
n_default = n_for(default_mde)

missed_power = power(business_mde, n_default)

weeks = int(np.ceil(2 * n_business / USERS_PER_WEEK))

conv_c = np.random.binomial(n_business, p0, 1_000) / n_business
conv_t = np.random.binomial(n_business, p0 + business_mde, 1_000) / n_business
se = np.sqrt(conv_c * (1 - conv_c) / n_business + conv_t * (1 - conv_t) / n_business)
sim_power = np.mean(np.abs(conv_t - conv_c) / se > z_alpha)
```

</details>

---

**Takeaway:** Threshold before calculator. The MDE you choose is the smallest effect you are willing to be blind to.

**Next:** [Task 08](https://github.com/dinbav/digital-experiments-course/blob/main/tasks/task-08.md) · [Open the worked demo in Colab](https://colab.research.google.com/github/dinbav/digital-experiments-course/blob/main/solutions/unit-08-power-mde-demo.ipynb)

**Further reading:** Kohavi, Tang & Xu, *Trustworthy Online Controlled Experiments* (2020), ch. 17 (the statistics behind experiments) and ch. 15 (ramping and duration).